# 07 · Administração: coleções, documentos, trechos

Tudo o que está guardado pode ser visto e controlado: **coleções** (bases de conhecimento com
nome), **documentos**, **trechos** e **memórias**. Este notebook usa a biblioteca; o mesmo vale
pela linha de comando, pelo HTTP, pelo MCP e pela página de administração local
(`pi-memory serve-http` → `http://127.0.0.1:8765/`).

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local]"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
from pi_memory import PiMemory

FIXTURES = Path("../../eval/fixtures")
PASTA = pasta_limpa("dados_07")
pm = PiMemory(path=PASTA, llm=lambda prompt: "")      # só administração: sem modelo de linguagem

## Documentos em coleções

Cada documento entra numa coleção (padrão `geral`). Também dá para criar um documento digitando
o texto.

In [3]:
pm.documents.add(FIXTURES / "contrato_exemplo.pdf", owner="empresa", collection="contratos")
pm.documents.add(FIXTURES / "vendas.xlsx", owner="empresa", collection="financeiro")
pm.documents.add(FIXTURES / "relatorio.docx", owner="empresa", collection="financeiro")
r = pm.documents.add_text("Procedimento de backup",
                          "O backup roda toda sexta às 22h. A cópia fica no servidor Exemplo.",
                          owner="empresa", collection="ti")
print(r["status"], r["document"]["source"], "→", r["document"]["collection"])
mostrar(pm.documents.collections("empresa"))

added Procedimento de backup.md → ti
{
  "collections": [
    {
      "name": "contratos",
      "documents": 1,
      "chunks": 20,
      "updated_at": "2026-11-10T16:52:01.763285Z"
    },
    {
      "name": "financeiro",
      "documents": 2,
      "chunks": 3,
      "updated_at": "2026-11-10T16:52:02.987794Z"
    },
    {
      "name": "ti",
      "documents": 1,
      "chunks": 1,
      "updated_at": "2026-11-10T16:52:03.088262Z"
    }
  ]
}


## Visão geral do dono

In [4]:
pm.memories.remember("O usuário é o responsável pelo financeiro", owner="empresa", topic="trabalho")
s = pm.stats("empresa")
print(s["memories"], "| documentos:", s["documents"], "| trechos:", s["chunks"])

{'total': 1, 'by_topic': {'trabalho': 1}} | documentos: 4 | trechos: 24


## Um documento e seus trechos; apagar um trecho

In [5]:
contrato = pm.documents.list("empresa", collection="contratos")["documents"][0]
doc = pm.documents.get(contrato["id"], "empresa")
print(doc["document"]["source"], "-", doc["document"]["chunks"], "trechos")
for c in doc["chunks"][:3]:
    print(f"  #{c['index']} {c['location']}: {c['text'][:60]}")

print(pm.documents.delete_chunk(doc["chunks"][0]["id"], "empresa"))
print("agora:", pm.documents.list("empresa", collection="contratos")["documents"][0]["chunks"], "trechos")

contrato_exemplo.pdf - 20 trechos
  #0 página 1: Página 1 Contrato de prestação de serviços entre a Empresa E
  #1 página 2: Página 2 Objeto: manutenção mensal de equipamentos de refrig
  #2 página 3: Página 3 As visitas técnicas acontecem na primeira segunda-f
{'deleted': 1}
agora: 19 trechos


## Busca restrita a uma coleção e exclusão de uma coleção inteira

In [6]:
for hit in pm.documents.search("quando roda o backup?", "empresa", collection="ti")["results"]:
    print(hit["citation"]["label"], "→", hit["text"][:70])

print(pm.documents.delete_collection("financeiro", "empresa"))
print([c["name"] for c in pm.documents.collections("empresa")["collections"]])
pm.close()

Procedimento de backup.md, seção Procedimento de backup → O backup roda toda sexta às 22h. A cópia fica no servidor Exemplo.


{'deleted_documents': 2, 'deleted_chunks': 3}
['contratos', 'ti']


## O mesmo pela linha de comando

Cada operação vira um comando; `--format text` mostra uma saída legível.

In [7]:
import subprocess, sys

def pi_memory(*args):
    saida = subprocess.run([sys.executable, "-m", "pi_memory.cli", "--path", PASTA, "--format", "text", *args],
                           capture_output=True, text=True, encoding="utf-8")
    print(saida.stdout.strip() or saida.stderr.strip())

pi_memory("collections", "list", "--owner", "empresa")
pi_memory("documents", "list", "--owner", "empresa", "--collection", "ti")

- contratos: 1 documento(s), 19 trecho(s)
- ti: 1 documento(s), 1 trecho(s)


- ti/Procedimento de backup.md  md  1 trecho(s)  (7daf26edec6d4214)


## O mesmo pelo servidor HTTP

O servidor serve a página de administração em `/` e as operações em `/v1/<operação>`. Aqui ele
sobe numa porta livre, só nesta máquina, e é chamado como qualquer outro programa chamaria.

In [8]:
import threading, urllib.request
from pi_memory.servers.http import make_server

pm = PiMemory(path=PASTA, llm=lambda prompt: "")
servidor = make_server(pm, port=0)
threading.Thread(target=servidor.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{servidor.server_address[1]}"

pedido = urllib.request.Request(url + "/v1/stats", data=json.dumps({"owner": "empresa"}).encode(),
                                headers={"Content-Type": "application/json"})
with urllib.request.urlopen(pedido) as resposta:
    mostrar(json.loads(resposta.read())["collections"])
with urllib.request.urlopen(url + "/") as resposta:
    pagina = resposta.read().decode()
print("página de administração:", len(pagina), "bytes;", "título:", pagina.split("<title>")[1].split("<")[0])

servidor.shutdown(); servidor.server_close(); pm.close()

[
  {
    "name": "contratos",
    "documents": 1,
    "chunks": 19,
    "updated_at": "2026-11-10T16:52:01.763285Z"
  },
  {
    "name": "ti",
    "documents": 1,
    "chunks": 1,
    "updated_at": "2026-11-10T16:52:03.088262Z"
  }
]
página de administração: 30836 bytes; título: pi-memory · administração


![Administração local: base de conhecimento](../../docs/img/admin-base-de-conhecimento.png)